In [1]:
%pip install pandas scipy matplotlib seaborn scikit-learn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import kendalltau
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel
from itertools import combinations

# New causal metrics calculation on the results from t-ecd

## T-ECD Marketplace

### PropCare mod, DLCE orig

In [106]:
def compute_cwu_at_k(df, k):
    total_weighted = 0
    total_weight = 0
    
    for _, group in df.groupby('idx_user'):
        n_items = len(group)
        actual_k = min(k, n_items)
        
        if actual_k == 0:
            continue
        
        top_k = group.head(actual_k)
        sum_effect = top_k['causal_effect'].sum()
        
        # CWU@K: weight = actual_k/k
        weight = actual_k / k
        total_weighted += weight * (sum_effect / actual_k)  # avg effect
        total_weight += weight
    
    return total_weighted / total_weight if total_weight > 0 else np.nan

In [76]:
def compute_rcau_at_k(df, k):
    """
    Rank-Completeness Adjusted Uplift
    Downweights users based on missing items in top-K
    """
    def rcau_per_user(group):
        n_items = len(group)
        actual_k = min(k, n_items)
        
        if actual_k == 0:
            return 0, 0
        
        top_k = group.head(actual_k)
        avg_uplift = top_k['causal_effect'].mean()
        
        # Completeness penalty: (actual_k / k)^2
        completeness_penalty = (actual_k / k) ** 2
        
        return avg_uplift * completeness_penalty, completeness_penalty
    
    weighted_sum = 0
    total_weight = 0
    
    for _, group in df.groupby('idx_user'):
        val, w = rcau_per_user(group)
        weighted_sum += val
        total_weight += w
    
    return weighted_sum / total_weight if total_weight > 0 else np.nan

In [92]:
def compute_dir_at_k(df, k, min_items=3, re_sort='no'):
    """
    DIR@K - Debiased Incremental Response at K
    
    Parameters:
    - df: DataFrame already sorted by ranking column
    - k: number of top items to consider
    - min_items: minimum items per user to include
    - re_sort: if True, re-sort by colname_estimate (for bounds)
    """
    df['propensity_estimate'] = df['propensity_estimate'].clip(0.01, 0.99)  # Avoid extreme propensities

    df['causal_effect_estimate'] = df['outcome'] * \
        (df['treated'] / df['propensity_estimate'] - \
            (1 - df['treated']) / (1 - df['propensity_estimate']))
    
    valid_effects = []
    
    for _, group in df.groupby('idx_user'):
        if len(group) >= min_items:
            if re_sort == 'max':
                # For theoretical bounds
                group_sorted = group.sort_values(by='causal_effect_estimate', ascending=False)
            elif re_sort == 'min':
                group_sorted = group.sort_values(by='causal_effect_estimate', ascending=True)
            else:
                # For actual ranking evaluation
                group_sorted = group
            
            top_k = group_sorted.head(k)
            valid_effects.append(top_k['causal_effect_estimate'].mean())
    
    total_users = df['idx_user'].nunique()
    coverage = len(valid_effects) / total_users if total_users > 0 else 0
    metric = np.mean(valid_effects) if valid_effects else np.nan

    for user_id, group in df.groupby('idx_user'):
        if len(group) >= 3:
            estimates = group['causal_effect_estimate']
            print(f"User {user_id}: min={estimates.min():.3f}, max={estimates.max():.3f}, mean={estimates.mean():.3f}")
            break

    # Also check global distribution
    print(df[df.groupby('idx_user')['idx_user'].transform('size') >= 3]['causal_effect_estimate'].describe())
    
    return metric, coverage

In [104]:
df = pd.read_csv('./results/default/mod/l/orig/df_sorted.csv')
df_sorted_main = df.sort_values(by=['idx_user', 'causal_effect'], ascending=False)
df_sorted_worst = df.sort_values(by=['idx_user', 'causal_effect'], ascending=True)
df_sorted_s = df.sort_values(by=['idx_user', 'pred'], ascending=False)
df_sorted_sf = df.sort_values(by=['idx_user', 'pred_freq'], ascending=False)
df_sorted_sfi = df.sort_values(by=['idx_user', 'pred_freqi'], ascending=False)
df_sorted_sfu = df.sort_values(by=['idx_user', 'pred_frequ'], ascending=False)
df_sorted_r = df.sort_values(by=['idx_user', 'relevance_estimate'], ascending=False)
df_sorted_pop = df.sort_values(by=['idx_user', 'popularity'], ascending=False)
df_sorted_pers_pop = df.sort_values(by=['idx_user', 'personal_popular'], ascending=False)

In [107]:
print("Max Coverage-Weighted Uplift@10: ", compute_cwu_at_k(df_sorted_main, 10))
print("Max Rank-Completeness Adjusted Uplift@10: ", compute_rcau_at_k(df_sorted_main, 10))
print("Max Sparsity-Adjusted DIR@10: ", compute_dir_at_k(df_sorted_main, 10, re_sort='max'))

Max Coverage-Weighted Uplift@10:  0.14608388000546174
Max Rank-Completeness Adjusted Uplift@10:  0.17976579644697002
User 8: min=-0.000, max=0.000, mean=0.000
count    7.927690e+06
mean     5.489423e-01
std      6.568390e+00
min     -9.991008e+00
25%      0.000000e+00
50%     -0.000000e+00
75%      0.000000e+00
max      1.000000e+02
Name: causal_effect_estimate, dtype: float64
Max Sparsity-Adjusted DIR@10:  (np.float64(1.755532611126544), 0.6499485941620848)


In [95]:
print("Min Coverage-Weighted Uplift@10: ", compute_cwu_at_k(df_sorted_worst, 10))
print("Min Rank-Completeness Adjusted Uplift@10: ", compute_rcau_at_k(df_sorted_worst, 10))
print("Min Sparsity-Adjusted DIR@10: ", compute_dir_at_k(df_sorted_worst, 10, re_sort='min'))

Min Coverage-Weighted Uplift@10:  -0.13376817615671202
Min Rank-Completeness Adjusted Uplift@10:  -0.17741871798700826
User 8: min=-0.000, max=0.000, mean=0.000
count    7.927690e+06
mean     5.489423e-01
std      6.568390e+00
min     -9.991008e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.000000e+02
Name: causal_effect_estimate, dtype: float64
Min Sparsity-Adjusted DIR@10:  (np.float64(0.0910012198976932), 0.6499485941620848)


In [96]:
print("Coverage-Weighted Uplift@10 for pred: ", compute_cwu_at_k(df_sorted_s, 10))
print("Rank-Completeness Adjusted Uplift@10 for pred: ", compute_rcau_at_k(df_sorted_s, 10))
print("Sparsity-Adjusted DIR@10 for pred: ", compute_dir_at_k(df_sorted_s, 10))

Coverage-Weighted Uplift@10 for pred:  0.007490995124585722
Rank-Completeness Adjusted Uplift@10 for pred:  0.0028750743665796495
User 8: min=-0.000, max=0.000, mean=0.000
count    7.927690e+06
mean     5.489423e-01
std      6.568390e+00
min     -9.991008e+00
25%      0.000000e+00
50%     -0.000000e+00
75%      0.000000e+00
max      1.000000e+02
Name: causal_effect_estimate, dtype: float64
Sparsity-Adjusted DIR@10 for pred:  (np.float64(0.45865937576662474), 0.6499485941620848)


In [98]:
print("Coverage-Weighted Uplift@10 for pred_freq: ", compute_cwu_at_k(df_sorted_sf, 10))
print("Rank-Completeness Adjusted Uplift@10 for pred_freq: ", compute_rcau_at_k(df_sorted_sf, 10))
print("Sparsity-Adjusted DIR@10 for pred_freq: ", compute_dir_at_k(df_sorted_sf, 10))

Coverage-Weighted Uplift@10 for pred_freq:  0.006488058324767518
Rank-Completeness Adjusted Uplift@10 for pred_freq:  0.0015949926735431231
User 8: min=-0.000, max=0.000, mean=0.000
count    7.927690e+06
mean     5.489423e-01
std      6.568390e+00
min     -9.991008e+00
25%      0.000000e+00
50%     -0.000000e+00
75%      0.000000e+00
max      1.000000e+02
Name: causal_effect_estimate, dtype: float64
Sparsity-Adjusted DIR@10 for pred_freq:  (np.float64(0.45652078285142506), 0.6499485941620848)


In [99]:
print("Coverage-Weighted Uplift@10 for pred_freqi: ", compute_cwu_at_k(df_sorted_sfi, 10))
print("Rank-Completeness Adjusted Uplift@10 for pred_freqi: ", compute_rcau_at_k(df_sorted_sfi, 10))
print("Sparsity-Adjusted DIR@10 for pred_freqi: ", compute_dir_at_k(df_sorted_sfi, 10))

Coverage-Weighted Uplift@10 for pred_freqi:  0.007184002660588798
Rank-Completeness Adjusted Uplift@10 for pred_freqi:  0.0024832496442898262
User 8: min=-0.000, max=0.000, mean=0.000
count    7.927690e+06
mean     5.489423e-01
std      6.568390e+00
min     -9.991008e+00
25%      0.000000e+00
50%     -0.000000e+00
75%      0.000000e+00
max      1.000000e+02
Name: causal_effect_estimate, dtype: float64
Sparsity-Adjusted DIR@10 for pred_freqi:  (np.float64(0.4586743776495573), 0.6499485941620848)


In [100]:
print("Coverage-Weighted Uplift@10 for pred_frequ: ", compute_cwu_at_k(df_sorted_sfu, 10))
print("Rank-Completeness Adjusted Uplift@10 for pred_frequ: ", compute_rcau_at_k(df_sorted_sfu, 10))
print("Sparsity-Adjusted DIR@10 for pred_frequ: ", compute_dir_at_k(df_sorted_sfu, 10))

Coverage-Weighted Uplift@10 for pred_frequ:  0.0065870728694826805
Rank-Completeness Adjusted Uplift@10 for pred_frequ:  0.001721368239837253
User 8: min=-0.000, max=0.000, mean=0.000
count    7.927690e+06
mean     5.489423e-01
std      6.568390e+00
min     -9.991008e+00
25%      0.000000e+00
50%     -0.000000e+00
75%      0.000000e+00
max      1.000000e+02
Name: causal_effect_estimate, dtype: float64
Sparsity-Adjusted DIR@10 for pred_frequ:  (np.float64(0.46076364723600627), 0.6499485941620848)


In [101]:
print("Coverage-Weighted Uplift@10 for relevance: ", compute_cwu_at_k(df_sorted_r, 10))
print("Rank-Completeness Adjusted Uplift@10 for relevance: ", compute_rcau_at_k(df_sorted_r, 10))
print("Sparsity-Adjusted DIR@10 for relevance: ", compute_dir_at_k(df_sorted_r, 10))

Coverage-Weighted Uplift@10 for relevance:  0.005093800884115323
Rank-Completeness Adjusted Uplift@10 for relevance:  -0.00018454460685631723
User 8: min=-0.000, max=0.000, mean=0.000
count    7.927690e+06
mean     5.489423e-01
std      6.568390e+00
min     -9.991008e+00
25%      0.000000e+00
50%     -0.000000e+00
75%      0.000000e+00
max      1.000000e+02
Name: causal_effect_estimate, dtype: float64
Sparsity-Adjusted DIR@10 for relevance:  (np.float64(0.29574274678496704), 0.6499485941620848)


In [102]:
print("Coverage-Weighted Uplift@10 for popularity: ", compute_cwu_at_k(df_sorted_pop, 10))
print("Rank-Completeness Adjusted Uplift@10 for popularity: ", compute_rcau_at_k(df_sorted_pop, 10))
print("Sparsity-Adjusted DIR@10 for popularity: ", compute_dir_at_k(df_sorted_pop, 10))

Coverage-Weighted Uplift@10 for popularity:  -0.011402306517757421
Rank-Completeness Adjusted Uplift@10 for popularity:  -0.021239076752125278
User 8: min=-0.000, max=0.000, mean=0.000
count    7.927690e+06
mean     5.489423e-01
std      6.568390e+00
min     -9.991008e+00
25%      0.000000e+00
50%     -0.000000e+00
75%      0.000000e+00
max      1.000000e+02
Name: causal_effect_estimate, dtype: float64
Sparsity-Adjusted DIR@10 for popularity:  (np.float64(0.4381676591472377), 0.6499485941620848)


In [97]:
print("Coverage-Weighted Uplift@10 for personal_popular: ", compute_cwu_at_k(df_sorted_pers_pop, 10))
print("Rank-Completeness Adjusted Uplift@10 for personal_popular: ", compute_rcau_at_k(df_sorted_pers_pop, 10))
print("Sparsity-Adjusted DIR@10 for personal_popular: ", compute_dir_at_k(df_sorted_pers_pop, 10))

Coverage-Weighted Uplift@10 for personal_popular:  -0.01152785127541086
Rank-Completeness Adjusted Uplift@10 for personal_popular:  -0.021399313714172714
User 8: min=-0.000, max=0.000, mean=0.000
count    7.927690e+06
mean     5.489423e-01
std      6.568390e+00
min     -9.991008e+00
25%      0.000000e+00
50%     -0.000000e+00
75%      0.000000e+00
max      1.000000e+02
Name: causal_effect_estimate, dtype: float64
Sparsity-Adjusted DIR@10 for personal_popular:  (np.float64(0.4423172956180856), 0.6499485941620848)
